# Synthetic HTML Generation + Evaluation

This notebook wraps the two scripts in a notebook workflow:
- generate synthetic HTML samples (`source.html` + `render.png`)
- optionally evaluate a running model server on those samples


In [ ]:
from pathlib import Path
import subprocess
import sys

from PIL import Image
from IPython.display import display

from vcoder.data.synth_html.dataset import generate_dataset


In [ ]:
# Generation settings
OUTPUT_DIR = Path("outputs/synth_html_nb")
NUM_SAMPLES = 30
SEED = 1337

# Evaluation settings
RUN_EVAL = False  # set True when your model server is running
EVAL_OUTPUT_DIR = Path("outputs/synth_eval_nb")
HOST = "localhost"
PORT = 8000
DEVICE = "cpu"  # "cuda" if available


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
generate_dataset(output_dir=OUTPUT_DIR, num_samples=NUM_SAMPLES, seed=SEED)
print(f"Generated dataset in: {OUTPUT_DIR}")
print(f"Index file: {OUTPUT_DIR / 'index.csv'}")


In [ ]:
# Preview first 3 rendered samples
sample_dirs = sorted([p for p in OUTPUT_DIR.glob("sample_*") if p.is_dir()])
for sample_dir in sample_dirs[:3]:
    png = sample_dir / "render.png"
    html = sample_dir / "source.html"
    print(sample_dir.name)
    print(f"  html: {html}")
    print(f"  png : {png}")
    display(Image.open(png))


## Optional: Evaluate Model Predictions

Requires an OpenAI-compatible VLM server running (for example on `localhost:8000`).
Set `RUN_EVAL = True` in the config cell before running this.

In [ ]:
if RUN_EVAL:
    EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        "-m",
        "vcoder.eval.evaluate_synthetic_html",
        "--dataset_dir", str(OUTPUT_DIR),
        "--output_dir", str(EVAL_OUTPUT_DIR),
        "--host", HOST,
        "--port", str(PORT),
        "--device", DEVICE,
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print(f"Evaluation complete. Report: {EVAL_OUTPUT_DIR / 'report.csv'}")
else:
    print("Skipping evaluation because RUN_EVAL is False.")


In [ ]:
# Quick paths summary
print("Generation output:", OUTPUT_DIR)
print("Generation index:", OUTPUT_DIR / "index.csv")
if RUN_EVAL:
    print("Evaluation output:", EVAL_OUTPUT_DIR)
    print("Evaluation report:", EVAL_OUTPUT_DIR / "report.csv")
    print("Gallery folder:", EVAL_OUTPUT_DIR / "gallery")
